In [2]:
%load_ext autoreload
%autoreload 2

# Import library

In [3]:

from sklearn.decomposition import PCA
from kmeans_pytorch import kmeans
from matplotlib.lines import Line2D
from sentence_transformers import SentenceTransformer, util , SimilarityFunction


import pandas as pd
import numpy as np
import regex as re
import networkx as nx
import itertools as it
import matplotlib.pyplot as plt
import pickle
import os
import torch
import seaborn as sns 
import nltk
import sys



/Users/gl665/PycharmProjects/Phrame_MentalMaps/.venv/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
2024-05-29 15:16:17.952130: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:
# Add the src directory to the sys.path
sys.path.append(os.path.abspath(os.path.join('..', 'src')))
print(sys.path)
import data_io

['/Library/Frameworks/Python.framework/Versions/3.12/lib/python312.zip', '/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12', '/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/lib-dynload', '', '/Users/gl665/PycharmProjects/Phrame_MentalMaps/.venv/lib/python3.12/site-packages', '/Users/gl665/PycharmProjects/Phrame_MentalMaps/src']


# Read data


## SYNCH Project 

In [5]:
new_dir = "/Users/gl665/PycharmProjects/Phrame_MentalMaps"
# Change the current working directory
os.chdir(new_dir)


my_data_RC = {}
folder_path = 'results/SYNCH/JSON_file/RC/'

cld_data = data_io.read_json(folder_path, my_data_RC)

load: FM 51220Clean.json
load: Passages Clean.json
load: full circle GBM 5 24 21clean.json
load: C 7 6 20 clean model.json
load: Wavi combined clean.json
load: drug justice model clean.json
load: DSS23.json
load: E clean model 572020.json
load: nat model clean2.json
load: FC Prov Clean.json
load: full circle clean.json
load: reentry clean model.json
load: DV justice model clean.json


In [22]:

for k, v in cld_data.items():
    for key, value in v[0].items():

        value = value.replace("\\", "")
        value = value.replace("/", " or ")
        value = value.lower()
        print(key, value )

1 abandonment
2 ability to navigate colonial spaces
3 acceptance of alcohol as coping behavior
4 acceptance of sexual assault
5 acceptance of violence
6 accessing healing resources
7 accessing victim resources
8 advocacy for equity and justice
9 alcohol misuse
10 attaining resources for success
11 attaining scholastic success
12 attaining success in colonial spaces
13 availability of healing resources
14 availability of victim support and resources
15 biracial or blood quantum racism
16 boarding school
17 bringing success to tribe
18 child abuse or aces
19 child in foster system
20 child removed from home
21 child witness ipv
22 coerced reproduction
23 colonial environment pushback
24 colonial environment or lakota identity clashes
25 colonial violence
26 community-level trauma
27 cultural fit for resources
28 cultural identity
29 cultural practices
30 cultural support
31 cultural values of honor and respect
32 culturally competent healing resources
33 culturally-competent spaces
34 cu

# Find semantic similary variable names 

In [6]:
# clean variables 

def clean_variable_name(variable_name):
    cleaned_name = variable_name.replace("\\", "")
    cleaned_name = cleaned_name.replace("/", " or ")
    cleaned_name = cleaned_name.lower()

    return cleaned_name


In [7]:
variable_names = []
for k, v in cld_data.items():

    cld_var_names = list(v[0].values())
    cld_var_names = clean_variable_names(cld_var_names)
    variable_names.append(cld_var_names)

variable_names = sum(variable_names, [])

variable_names = np.sort(list(set(variable_names)))


variable_names
print("number of originial unique nodes: ", len(variable_names))
# print all list of variable names


number of originial unique nodes:  1021


In [8]:
# Embed some sentences
# variable_names = [
#     'customer_age', 'client_age', 'age_of_customer', 
#     'customer_id', 'client_id', 'id_of_customer', 
#     'purchase_date', 'date_of_purchase', 'transaction_date', 'customer__age'
# ]

In [9]:
# Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-mpnet-base-v2")

embeddings = model.encode(variable_names)

similarities = model.similarity(embeddings, embeddings)

# TODO: Finetune the sentence transformer model to improve performance of the model. 
# # Change the similarity function to Manhattan distance
# model.similarity_fn_name = SimilarityFunction.MANHATTAN
# print(model.similarity_fn_name)
# # => "manhattan"

# similarities = model.similarity(embeddings, embeddings)
# print(similarities)


/Users/gl665/PycharmProjects/Phrame_MentalMaps/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [10]:
# Output the pairs with their score
for idx_i, sentence1 in enumerate(variable_names):
    print(sentence1)
    for idx_j, sentence2 in enumerate(variable_names):
        print(f" - {sentence2: <30}: {similarities[idx_i][idx_j]:.4f}")
        break 

abandonment
 - abandonment                   : 1.0000
ability for health insurance to pay for care
 - abandonment                   : 0.1021
ability for victims to recieve housing resources
 - abandonment                   : 0.2522
ability to address trauma
 - abandonment                   : 0.2151
ability to afford birth control
 - abandonment                   : 0.1228
ability to be self sustaining
 - abandonment                   : 0.2071
ability to fufil dss requests
 - abandonment                   : 0.0429
ability to fufill court requirements
 - abandonment                   : 0.1274
ability to move from detox to treatment
 - abandonment                   : 0.1169
ability to navigate colonial spaces
 - abandonment                   : 0.1823
ability to pay for treatment
 - abandonment                   : 0.1649
ability to pay for tx
 - abandonment                   : 0.0794
ability to qualify for extra ss help
 - abandonment                   : 0.0388
ability to schedule  or  time

In [11]:

# Threshold for considering names as similar
threshold = 0.7

# Group semantically similar names
groups = []
visited = set()

for idx, name in enumerate(variable_names):
    if name not in visited:
        similar_names = [name]
        for jdx, similarity in enumerate(similarities[idx]):
            
            if jdx != idx and similarity >= threshold:
                similar_names.append(variable_names[jdx])
                visited.add(variable_names[jdx])
        groups.append(similar_names)
        visited.update(similar_names)


In [12]:
print("numbers of final variables after grouping: ", len(groups))


numbers of final variables after grouping:  631


In [13]:
# Print grouped variable names
for i, group in enumerate(groups):
    if len(group) > 1:
        print(f"Group {i + 1}: {group}")

Group 2: ['ability for health insurance to pay for care', 'ability to pay for treatment', 'access to health insurance', 'access to medical care', 'affordability to medical care', 'having health insurance']
Group 3: ['ability for victims to recieve housing resources', 'access to housing', 'availability of victim shelter housing', 'use of shelter housing']
Group 4: ['ability to address trauma', 'unresolved trauma']
Group 5: ['ability to afford birth control', 'access to birth control', 'affordability of birth control', 'availability of birth control', 'health insurance benefits access for birth control', 'traditional insurance qualifications for birth control']
Group 6: ['ability to be self sustaining', 'self-sufficiency', 'self-sustainment']
Group 7: ['ability to fufil dss requests', 'being able to comply with dss']
Group 8: ['ability to fufill court requirements', 'court based requirements', 'court requirements fufilled']
Group 16: ['acceptability of birth control', 'affordability of b

In [14]:
# create a dictionary to swap the words that are bundled in the group. 
swap_dict = {}

for group in groups:
    if len(group) > 1:
        swap_dict[group[0]] = group
swap_dict

{'ability for health insurance to pay for care': ['ability for health insurance to pay for care',
  'ability to pay for treatment',
  'access to health insurance',
  'access to medical care',
  'affordability to medical care',
  'having health insurance'],
 'ability for victims to recieve housing resources': ['ability for victims to recieve housing resources',
  'access to housing',
  'availability of victim shelter housing',
  'use of shelter housing'],
 'ability to address trauma': ['ability to address trauma',
  'unresolved trauma'],
 'ability to afford birth control': ['ability to afford birth control',
  'access to birth control',
  'affordability of birth control',
  'availability of birth control',
  'health insurance benefits access for birth control',
  'traditional insurance qualifications for birth control'],
 'ability to be self sustaining': ['ability to be self sustaining',
  'self-sufficiency',
  'self-sustainment'],
 'ability to fufil dss requests': ['ability to fufil ds

In [15]:
cld_data['C 7 6 20 clean model.json'][0]

{1: 'ability for health insurance to pay for care',
 2: 'ability to self pay',
 3: 'acceptance of medicaid as payment',
 4: 'access to getting prenatal care',
 5: 'access to health insurance',
 6: 'access to reproductive healthcare',
 7: 'accuracy of screening',
 8: 'agency to use birth control',
 9: 'aian/white staff ratio',
 10: 'alcohol exposed pregnancy',
 11: 'alcohol misuse',
 12: 'attitude towards large families/pro pregnancy',
 13: 'availability of birth control',
 14: 'availability of prenatal healthcare',
 15: 'availability of resources for reproductive health',
 16: 'birth control information/education',
 17: 'birth control use',
 18: 'child home functioning',
 19: 'child removed from maternal custody',
 20: 'child witness ipv',
 21: 'childbirth',
 22: 'children being provided healthcare',
 23: 'client agrees to screen',
 24: 'clients in need',
 25: 'clinic screen for drugs',
 26: 'comfort/fit for aian women',
 27: 'continued medical checkups for new mothers',
 28: 'cost of 

In [16]:
cld_data_new['C 7 6 20 clean model.json'][0]

NameError: name 'cld_data_new' is not defined

In [17]:
cld_data_new = cld_data.copy()

cld_data_new

{'FM 51220Clean.json': [{1: 'abandonment',
   2: 'ability to navigate colonial spaces',
   3: 'acceptance of alcohol as coping behavior',
   4: 'acceptance of sexual assault',
   5: 'acceptance of violence',
   6: 'accessing healing resources',
   7: 'accessing victim resources',
   8: 'advocacy for equity and justice',
   9: 'alcohol misuse',
   10: 'attaining resources for success',
   11: 'attaining scholastic success',
   12: 'attaining success in colonial spaces',
   13: 'availability of healing resources',
   14: 'availability of victim support and resources',
   15: 'biracial/blood quantum racism',
   16: 'boarding school',
   17: 'bringing success to tribe',
   18: 'child abuse/aces',
   19: 'child in foster system',
   20: 'child removed from home',
   21: 'child witness ipv',
   22: 'coerced reproduction',
   23: 'colonial environment pushback',
   24: 'colonial environment/lakota identity clashes',
   25: 'colonial violence',
   26: 'community-level trauma',
   27: 'cultural

In [62]:
# swap the orignial variable name to the grouped variable name using the swap_dict 
cld_data_new = cld_data.copy()

cld_data_new['']
for k, v in cld_data_new.items():
    for key, value in v[0].items():
        print(key, value    )
        for merged_key, original_value in swap_dict.items():
            print(value, original_value)
            if value in original_value:
                v[0][key] = merged_key


In [61]:
swap_dict.keys()

dict_keys(['ability for health insurance to pay for care', 'ability for victims to recieve housing resources', 'ability to address trauma', 'ability to afford birth control', 'ability to be self sustaining', 'ability to fufil dss requests', 'ability to fufill court requirements', 'acceptability of birth control', 'acceptance of alcohol as coping behavior', 'acceptance of medicaid as payment', 'acceptance of violence', 'access to abortion', 'access to affordable housing', 'access to daycare', 'access to drugs', 'access to getting prenatal care', 'access to opioid alternatives', 'access to positive social support', 'access to re-entry resources', 'access to sober spaces', 'accessing alcohol use treatemnt', 'accessing healing resources', 'accessing mh therapy', 'accessing victim resources', 'accuracy of assessment', 'achieving sobriety', 'advocate care to victims', 'advocate case load', 'affordability of therapy', 'agency to use birth control', 'aian experienced discrimination by police',